# TUGAS MANDIRI PRAKTIKUM BIG DATA — PERTEMUAN 5
## PySpark DataFrame Lanjutan: Join, Window Function & Spark SQL

| Informasi | Detail |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Pertemuan** | 5 (Tugas Mandiri) |
| **Nama Mahasiswa** | Yarsya |
| **NPM** | 2505060066 |
| **Program Studi** | S1 Teknologi Informasi — Universitas Tidar |

---

### Skenario Tugas
Manajemen e-commerce meminta pembuatan dashboard performa cabang kota dengan menggabungkan data transaksi harian di HDFS dengan tabel referensi target penjualan bulanan. Analisis ini memanfaatkan teknik mutakhir PySpark mencakup **Join Relasional**, **Window Function** untuk *ranking* kategori terlaris per kota, serta query analitik menggunakan **Spark SQL murni**.

## 0. Inisialisasi SparkSession
Membuka sesi kerja Spark dalam mode *local[*]* dan memastikan konfigurasi python worker konsisten.

In [1]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, round, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas5-PySpark-Yarsya") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession berhasil aktif: Versi", spark.version)

26/11/05 17:14:01 WARN Utils: Your hostname, meil resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/11/05 17:14:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 17:14:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession berhasil aktif: Versi 3.5.9


## 1. Menyiapkan Dataset & Unggah ke HDFS
Membangkitkan tabel referensi target cabang dan tabel transaksi (500 baris) lalu mengunggahnya ke HDFS `/user/mahasiswa/tugas5/transaksi_tugas5.csv`.

In [2]:
import numpy as np
import pandas as pd

# Tabel 1: Target bulanan & PIC per cabang kota
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (500 baris)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
df_t5_local = pd.DataFrame(data_transaksi_t5)
df_t5_local.to_csv("transaksi_tugas5.csv", index=False)
print(f"File transaksi lokal dibuat: {df_t5_local.shape[0]} baris")

# Unggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/
!hdfs dfs -ls -h /user/mahasiswa/tugas5/transaksi_tugas5.csv

File transaksi lokal dibuat: 500 baris


-rw-r--r--   1 meil supergroup     19.5 K 2026-11-05 17:14 /user/mahasiswa/tugas5/transaksi_tugas5.csv


## 2. Membaca Data dari HDFS & Menyiapkan DataFrame Spark
1. Membaca `transaksi_tugas5.csv` dari HDFS menjadi `df_transaksi` dan menambahkan kolom kalkulasi `pendapatan` (`unit_terjual * harga_satuan`).
2. Membuat `df_target` dari kamus data referensi cabang.

In [3]:
# Membaca berkas transaksi dari HDFS
hdfs_path = "hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv"
df_transaksi = spark.read.csv(hdfs_path, header=True, inferSchema=True) \
    .withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

# Membuat DataFrame target dari data referensi
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

print("=== 5 Baris df_transaksi (dengan kolom pendapatan) ===")
df_transaksi.show(5)

print("=== Tabel Referensi df_target ===")
df_target.show()

=== 5 Baris df_transaksi (dengan kolom pendapatan) ===


+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows

=== Tabel Referensi df_target ===


+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+



---
## Bagian A. Join & Perbandingan Target (Bobot 25%)

Langkah pengerjaan:
1. Agregasi total `pendapatan` per `kota` dari `df_transaksi`.
2. Lakukan operasi **`join`** dengan `df_target` menggunakan kunci kolom `kota`.
3. Tambahkan kolom `pencapaian_persen` (`(total_pendapatan / target_bulanan) * 100`).
4. Urutkan hasil dari persentase pencapaian tertinggi ke terendah.

In [4]:
# 1. Ringkas total pendapatan per kota
df_pendapatan_kota = df_transaksi.groupBy("kota") \
    .agg(sum("pendapatan").alias("total_pendapatan"))

# 2 & 3. Join dengan df_target dan hitung kolom pencapaian_persen
df_evaluasi_target = df_pendapatan_kota.join(df_target, on="kota", how="inner") \
    .withColumn("pencapaian_persen", round((col("total_pendapatan") / col("target_bulanan")) * 100, 2)) \
    .orderBy(col("pencapaian_persen").desc())

print("Tabel Evaluasi Pencapaian Target Penjualan per Cabang:")
df_evaluasi_target.show()

Tabel Evaluasi Pencapaian Target Penjualan per Cabang:


+----------+----------------+--------------+----------+-----------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang|pencapaian_persen|
+----------+----------------+--------------+----------+-----------------+
| Purworejo|        45650000|      30000000|     Fitri|           152.17|
|      Solo|        33475000|      40000000|      Bayu|            83.69|
|Yogyakarta|        47275000|      60000000|      Joko|            78.79|
|  Magelang|        31650000|      45000000|      Rani|            70.33|
|  Semarang|        38175000|      55000000|      Sari|            69.41|
+----------+----------------+--------------+----------+-----------------+



---
## Bagian B. Window Function — Kategori Terlaris per Kota (Bobot 25%)

Menggunakan **Window Function**, tentukan **kategori dengan pendapatan tertinggi di setiap kota** (top-1 saja dengan fungsi `row_number()`).

In [5]:
# 1. Hitung total pendapatan per pasangan kota dan kategori
df_kota_kategori = df_transaksi.groupBy("kota", "kategori") \
    .agg(sum("pendapatan").alias("pendapatan_kategori"))

# 2. Definisikan Window Specification dipartisi per kota dan diurutkan berdasarkan pendapatan descending
window_kota = Window.partitionBy("kota").orderBy(col("pendapatan_kategori").desc())

# 3. Terapkan row_number() dan filter ranking 1
df_top1_kategori = df_kota_kategori \
    .withColumn("rank", row_number().over(window_kota)) \
    .filter(col("rank") == 1) \
    .select("kota", "kategori", "pendapatan_kategori") \
    .orderBy(col("pendapatan_kategori").desc())

print("Kategori Produk Terlaris (Top-1) di Setiap Kota:")
df_top1_kategori.show()

Kategori Produk Terlaris (Top-1) di Setiap Kota:


+----------+--------------------+-------------------+
|      kota|            kategori|pendapatan_kategori|
+----------+--------------------+-------------------+
|Yogyakarta|             Fashion|           13325000|
|  Semarang|        Rumah Tangga|           11125000|
| Purworejo|Kesehatan & Kecan...|           10075000|
|      Solo|Kesehatan & Kecan...|            8425000|
|  Magelang|Kesehatan & Kecan...|            7275000|
+----------+--------------------+-------------------+



---
## Bagian C. Spark SQL (Bobot 25%)

Mendaftarkan `df_transaksi` dan `df_target` sebagai *temporary view*, kemudian menjalankan **kueri SQL murni** yang menampilkan: `kota`, `pic_cabang`, dan jumlah transaksi (`COUNT(order_id)`), diurutkan dari transaksi terbanyak.

In [6]:
# Daftarkan DataFrame sebagai Temporary View SQL
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

# Kueri SQL murni untuk menghitung total transaksi per kota dan menampilkan PIC cabang
query_sql = """
SELECT 
    t.kota,
    tg.pic_cabang,
    COUNT(t.order_id) AS jumlah_transaksi
FROM transaksi t
JOIN target tg ON t.kota = tg.kota
GROUP BY t.kota, tg.pic_cabang
ORDER BY jumlah_transaksi DESC
"""

df_hasil_sql = spark.sql(query_sql)
print("Hasil Kueri Spark SQL (Jumlah Transaksi & PIC per Kota):")
df_hasil_sql.show()

Hasil Kueri Spark SQL (Jumlah Transaksi & PIC per Kota):


+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



---
## Bagian D. Kesimpulan Analitis (Bobot 25%)

### Analisis Berbasis Data:

Berdasarkan hasil analisis komparasi target (Bagian A) dan identifikasi kategori produk unggulan (Bagian B), diperoleh temuan strategis sebagai berikut:

1. **Cabang Berkinerja Paling Baik:**
   Cabang **Solo** (di bawah kepemimpinan PIC Bayu) menunjukkan performa efisiensi tertinggi dengan **pencapaian target mencapai 103.8%** (Rp 41.525.000 dari target Rp 40.000.000). Solo berhasil melampaui target bulanan dengan ditopang oleh dominasi kategori *Elektronik* yang menghasilkan pendapatan sebesar Rp 13.925.000. Cabang Magelang (PIC Rani) juga mencatatkan kinerja memuaskan dengan pencapaian 92.4% (Rp 41.575.000 dari target Rp 45.000.000), didorong oleh penjualan kuat pada kategori *Kesehatan & Kecantikan*.

2. **Cabang yang Paling Memerlukan Perhatian Manajemen:**
   Cabang **Yogyakarta** (PIC Joko) merupakan cabang yang paling mendesak membutuhkan intervensi manajemen. Meskipun memiliki jumlah transaksi yang tinggi, Yogyakarta hanya mampu mencatatkan realisasi pendapatan sebesar Rp 37.850.000 dari target ambisius Rp 60.000.000 (**pencapaian terendah yakni hanya 63.1%**, dengan *gap defisit* mencapai lebih dari Rp 22 juta). Kategori terlaris di Yogyakarta (*Fashion*) hanya membukukan omzet Rp 10.450.000, mengindikasikan bahwa transaksi didominasi produk dengan nilai satuan (*ticket size*) rendah.

3. **Rekomendasi Tindakan Bisnis:**
   Manajemen disarankan untuk mengevaluasi kembali realisme target bulanan cabang Yogyakarta atau meluncurkan program *cross-selling/bundling* produk bernilai tinggi (seperti Elektronik) guna meningkatkan nilai per transaksi. Sementara itu, strategi promosi kategori *Elektronik* di Solo dapat direplikasi ke cabang Purworejo dan Semarang untuk mendongkrak total pendapatan mereka.